# Products with the Thinnest Sales vs. Purchase Margin

Join `adventurework.gold.fact_sales` and `adventurework.gold.fact_purchases` on `ProductID` and find the 5 products where the gap between average sales price and average purchase price is smallest.

In [0]:
WITH sales_by_product AS (
  SELECT
    ProductID,
    AVG(UnitPrice) AS avg_sales_price
  FROM adventurework.gold.fact_sales
  GROUP BY ProductID
),
purchases_by_product AS (
  SELECT
    ProductID,
    AVG(UnitPrice) AS avg_purchase_price
  FROM adventurework.gold.fact_purchases
  GROUP BY ProductID
)
SELECT
  dim_p.Name AS product_name,
  s.ProductID,
  ROUND(s.avg_sales_price, 2) AS avg_sales_price,
  ROUND(pur.avg_purchase_price, 2) AS avg_purchase_price,
  ROUND(s.avg_sales_price - pur.avg_purchase_price, 2) AS margin
FROM sales_by_product AS s
INNER JOIN purchases_by_product AS pur
  ON s.ProductID = pur.ProductID
INNER JOIN adventurework.gold.dim_product AS dim_p
  ON s.ProductID = dim_p.ProductID
ORDER BY margin ASC
LIMIT 5;

product_name,ProductID,avg_sales_price,avg_purchase_price,margin
Front Brakes,948,63.88,82.83,-18.96
Rear Brakes,907,63.90,82.83,-18.93
ML Road Tire,932,24.99,39.46,-14.47
HL Mountain Pedal,937,48.57,62.99,-14.42
Touring Pedal,941,48.59,62.99,-14.40


In [0]:
WITH sales_monthly AS (
  SELECT
    ProductID,
    date_trunc('month', ModifiedDate) AS sales_month, -- fact_sales has no order date; ModifiedDate is the best available proxy
    SUM(OrderQty) AS units_sold
  FROM adventurework.gold.fact_sales
  GROUP BY ProductID, sales_month
),
purchases_monthly AS (
  SELECT
    ProductID,
    date_trunc('month', OrderDate) AS purchase_month,
    SUM(OrderQty) AS units_purchased
  FROM adventurework.gold.fact_purchases
  GROUP BY ProductID, purchase_month
),
monthly AS (
  -- FULL OUTER JOIN so months with only sales or only purchases are still visible
  SELECT
    COALESCE(s.ProductID, p.ProductID) AS ProductID,
    COALESCE(s.sales_month, p.purchase_month) AS month,
    COALESCE(s.units_sold, 0) AS units_sold,
    COALESCE(p.units_purchased, 0) AS units_purchased,
    COALESCE(s.units_sold, 0) - COALESCE(p.units_purchased, 0) AS net_units
  FROM sales_monthly AS s
  FULL OUTER JOIN purchases_monthly AS p
    ON s.ProductID = p.ProductID
   AND s.sales_month = p.purchase_month
)
SELECT
  dim_p.Name AS product_name,
  m.ProductID,
  ROUND(AVG(m.units_sold), 1) AS avg_units_sold_monthly,
  ROUND(AVG(m.units_purchased), 1) AS avg_units_purchased_monthly,
  SUM(m.net_units) AS total_net_units,
  ROUND(100.0 * COUNT_IF(m.units_sold > m.units_purchased) / COUNT(*), 0) AS pct_months_selling_faster
FROM monthly AS m
INNER JOIN adventurework.gold.dim_product AS dim_p
  ON m.ProductID = dim_p.ProductID
GROUP BY m.ProductID, dim_p.Name
HAVING SUM(m.net_units) > 0
ORDER BY total_net_units DESC, pct_months_selling_faster DESC
LIMIT 10;

product_name,ProductID,avg_units_sold_monthly,avg_units_purchased_monthly,total_net_units,pct_months_selling_faster
AWC Logo Cap,712,218.7,16.6,7681,97
"Sport-100 Helmet, Black",708,171.9,0.7,6507,100
"Long-Sleeve Logo Jersey, L",715,173.5,2.6,6492,100
"Sport-100 Helmet, Blue",711,177.4,7.9,6443,100
"Sport-100 Helmet, Red",707,164.9,0.7,6241,100
Water Bottle - 30 oz.,870,486.8,107.1,5315,93
"Long-Sleeve Logo Jersey, M",714,95.7,9.2,3286,97
Bike Wash - Dissolver,877,237.1,10.7,3169,93
"Classic Vest, S",864,303.4,89.3,2997,93
"Mountain-200 Black, 38",782,119.1,0.0,2977,100
